### Data Cleaning

#### Importing Libraries & Loading Data Files

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
project_root = Path(".").resolve().parent
raw_dir = project_root / "data" / "raw"
borough_files = sorted(raw_dir.glob("*_link_*.csv"))

df = pd.concat(
    [pd.read_csv(fp) for fp in borough_files],
    ignore_index=True,
)

#### Data

In [3]:
df

,priceper,year,dateoftransfer,propertytype,duration,price,postcode,lad23cd,transactionid,lmk_key,tfarea,numberrooms,classt,CURRENT_ENERGY_EFFICIENCY,POTENTIAL_ENERGY_EFFICIENCY,CONSTRUCTION_AGE_BAND
0,1115.384615,1999,1999-10-04,F,L,58000,IG11 0XN,E09000002,{C30CDD8C-3FA0-4259-ABE4-CA992C1AC593},126980681552014061013414193040742,52.0,3.0,12,78,81,England and Wales: 1991-1995
1,2942.307692,2014,2014-09-26,F,L,153000,IG11 0XN,E09000002,{10382CFD-B786-4103-A1C0-B787ED5944FB},126980681552014061013414193040742,52.0,3.0,12,78,81,England and Wales: 1991-1995
2,2644.230769,2004,2004-10-29,F,L,137500,IG11 0XN,E09000002,{8276E0B3-FDE3-41F4-8DE5-2E589472C69B},126980681552014061013414193040742,52.0,3.0,12,78,81,England and Wales: 1991-1995
3,902.884615,1997,1997-01-24,F,L,46950,IG11 0XN,E09000002,{2A289E9D-9E69-CDC8-E050-A8C063054829},126980681552014061013414193040742,52.0,3.0,12,78,81,England and Wales: 1991-1995
4,4206.349206,2023,2023-11-28,F,L,265000,IG11 7GW,E09000002,{152AB733-B73B-E651-E063-4704A8C061D9},5a6e6b6a24c2a840e15b66b8abcb70c3d09b4096b2f5e2...,63.0,3.0,12,76,79,England and Wales: 1996-2002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2806759,7920.000000,2014,08/01/2014,T,F,594000,W10 4AS,E09000033,{535389DD-E9D8-4BC1-92E1-FD6D7F14E472},9.4983E+32,75.0,4.0,11,60,80,England and Wales: 1900-1929
2806760,12900.013330,2024,12/07/2024,T,F,967501,W10 4AS,E09000033,{2131FCF6-462C-86E8-E063-4804A8C0372B},9.4983E+32,75.0,4.0,11,60,80,England and Wales: 1900-1929
2806761,4055.944056,2003,30/01/2003,F,L,174000,W9 3PG,E09000033,{9E9DAC1F-D8D9-4DD2-9F35-EE3E90377051},4.10972E+32,42.9,2.0,11,80,84,England and Wales: 1930-1949
2806762,2995.337995,2001,20/07/2001,F,L,128500,W9 3PG,E09000033,{626B9E83-4BD3-420D-BCC0-627A049625E7},4.10972E+32,42.9,2.0,11,80,84,England and Wales: 1930-1949


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2806764 entries, 0 to 2806763
Data columns (total 16 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   priceper                     float64
 1   year                         int64  
 2   dateoftransfer               object 
 3   propertytype                 object 
 4   duration                     object 
 5   price                        int64  
 6   postcode                     object 
 7   lad23cd                      object 
 8   transactionid                object 
 9   lmk_key                      object 
 10  tfarea                       float64
 11  numberrooms                  float64
 12  classt                       int64  
 13  CURRENT_ENERGY_EFFICIENCY    int64  
 14  POTENTIAL_ENERGY_EFFICIENCY  int64  
 15  CONSTRUCTION_AGE_BAND        object 
dtypes: float64(3), int64(5), object(8)
memory usage: 342.6+ MB


In [4]:
# Restricting to Year 2008 and onwards

df = df[df['year'] >= 2008]

In [5]:
# Keeping price range between 50k and 10M

df = df[df['price'].between(50000, 10000000)]

In [5]:
# Keeping floor area between 15 and 500 square metres

df = df[df['tfarea'].between(15, 500)]

In [7]:
# Fixing date values

df["dateoftransfer"] = pd.to_datetime(df["dateoftransfer"], format="mixed", dayfirst=True)

C:\Users\Abdul Qudus\AppData\Local\Temp\ipykernel_32896\2964512574.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["dateoftransfer"] = pd.to_datetime(df["dateoftransfer"], format="mixed", dayfirst=True)


In [10]:
df['dateoftransfer']

0         1999-10-04
1         2014-09-26
2         2004-10-29
3         1997-01-24
4         2023-11-28
             ...    
2806759   2014-01-08
2806760   2024-07-12
2806761   2003-01-30
2806762   2001-07-20
2806763   2006-08-09
Name: dateoftransfer, Length: 2801043, dtype: datetime64[ns]

In [11]:
# Resetting Index

df = df.reset_index(drop=True)

In [16]:
# Fixing Categorical Columns

df['CONSTRUCTION_AGE_BAND'] = df['CONSTRUCTION_AGE_BAND'].replace(
    ['INVALID!', 'NO DATA!'], 'unknown').fillna('unknown')

In [17]:
# Adding month column

df['month'] = df['dateoftransfer'].dt.month

In [18]:
# Adding Log Price column

df['log_price'] = np.log10(df['price'])

In [20]:
# Exploring Construction Age Band

df['CONSTRUCTION_AGE_BAND'].value_counts(ascending=False)

CONSTRUCTION_AGE_BAND
England and Wales: 1900-1929       603315
England and Wales: 1930-1949       578342
England and Wales: before 1900     360642
England and Wales: 1950-1966       244147
unknown                            211185
England and Wales: 1967-1975       172295
England and Wales: 1983-1990       148720
England and Wales: 1996-2002       142508
England and Wales: 2003-2006        92948
England and Wales: 1976-1982        89185
England and Wales: 1991-1995        85007
England and Wales: 2007 onwards     38031
England and Wales: 2007-2011        10174
2021                                 5884
2022                                 5116
England and Wales: 2012 onwards      3435
2019                                 3223
2020                                 2825
2018                                 1647
2023                                 1072
2017                                  645
2013                                  193
2016                                  168
2010        

In [21]:
df

,priceper,year,dateoftransfer,propertytype,duration,price,postcode,lad23cd,transactionid,lmk_key,tfarea,numberrooms,classt,CURRENT_ENERGY_EFFICIENCY,POTENTIAL_ENERGY_EFFICIENCY,CONSTRUCTION_AGE_BAND,month,log_price
0,1115.384615,1999,1999-10-04,F,L,58000,IG11 0XN,E09000002,{C30CDD8C-3FA0-4259-ABE4-CA992C1AC593},126980681552014061013414193040742,52.0,3.0,12,78,81,England and Wales: 1991-1995,10,4.763428
1,2942.307692,2014,2014-09-26,F,L,153000,IG11 0XN,E09000002,{10382CFD-B786-4103-A1C0-B787ED5944FB},126980681552014061013414193040742,52.0,3.0,12,78,81,England and Wales: 1991-1995,9,5.184691
2,2644.230769,2004,2004-10-29,F,L,137500,IG11 0XN,E09000002,{8276E0B3-FDE3-41F4-8DE5-2E589472C69B},126980681552014061013414193040742,52.0,3.0,12,78,81,England and Wales: 1991-1995,10,5.138303
3,902.884615,1997,1997-01-24,F,L,46950,IG11 0XN,E09000002,{2A289E9D-9E69-CDC8-E050-A8C063054829},126980681552014061013414193040742,52.0,3.0,12,78,81,England and Wales: 1991-1995,1,4.671636
4,4206.349206,2023,2023-11-28,F,L,265000,IG11 7GW,E09000002,{152AB733-B73B-E651-E063-4704A8C061D9},5a6e6b6a24c2a840e15b66b8abcb70c3d09b4096b2f5e2...,63.0,3.0,12,76,79,England and Wales: 1996-2002,11,5.423246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2801038,7920.000000,2014,2014-01-08,T,F,594000,W10 4AS,E09000033,{535389DD-E9D8-4BC1-92E1-FD6D7F14E472},9.4983E+32,75.0,4.0,11,60,80,England and Wales: 1900-1929,1,5.773786
2801039,12900.013330,2024,2024-07-12,T,F,967501,W10 4AS,E09000033,{2131FCF6-462C-86E8-E063-4804A8C0372B},9.4983E+32,75.0,4.0,11,60,80,England and Wales: 1900-1929,7,5.985651
2801040,4055.944056,2003,2003-01-30,F,L,174000,W9 3PG,E09000033,{9E9DAC1F-D8D9-4DD2-9F35-EE3E90377051},4.10972E+32,42.9,2.0,11,80,84,England and Wales: 1930-1949,1,5.240549
2801041,2995.337995,2001,2001-07-20,F,L,128500,W9 3PG,E09000033,{626B9E83-4BD3-420D-BCC0-627A049625E7},4.10972E+32,42.9,2.0,11,80,84,England and Wales: 1930-1949,7,5.108903


In [22]:
# Saving DataFrame
processed_dir = project_root /"data" /"processed"
out_path = processed_dir/'london_housing_2008_24.parquet'
df.to_parquet(out_path, index=False)